In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from glob import glob
from playwright.sync_api import sync_playwright

# Aplicar configuraciones de visualización total de Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

from pathlib import Path
import glob

import warnings
warnings.filterwarnings("ignore")
import plotly.express as px
import func_graff as graff

**Insumos**

In [2]:
periodo_analisis = "202606"
ruta = r"bd\bd_productividad.csv"
ruta_velocdidad = r"bd\bd_velocidad.csv"

In [3]:
#TIPO DE VENTA: LAST MILE + ECOMMERCE + POS

df_prod = pd.read_csv(ruta)
df_prod1 = df_prod[df_prod["GOS"].isin(['Fátima L', 'Sofía V', 'Sebastián S', 'Hernán P', 'María R'])][['Tienda', 'Periodo', 'GOS', 'GOR', 'Prod', 'VtaNeta', 'JEq',
       'ProdMeta', 'VtaNetaMeta', 'JEqMeta', 'ProdAA', 'VtaNetaAA', 'JEqAA']]


In [4]:
df_prod1.head()

,Tienda,Periodo,GOS,GOR,Prod,VtaNeta,JEq,ProdMeta,VtaNetaMeta,JEqMeta,ProdAA,VtaNetaAA,JEqAA
1,1198 Cusco Wanchaq - PVH,202501,Sofía V,Joe H,72841.674113,2380725.0,32.683555,64571.915303,2126328.0,36.869355,61497.0,1966223.0,31.972641
2,1199 San Miguel - PVH,202501,Sebastián S,Vik E,73057.259950,5564635.0,76.168138,71183.375077,5578169.0,78.173246,67794.0,5172368.0,76.295719
3,1593 Aventura SJL - PVH,202501,Sebastián S,Rodolfo O,99824.851993,5440157.0,54.497023,66842.964879,4627072.0,81.387133,63660.0,4339531.0,68.167345
4,2638 SJB Ayacucho - PVH,202501,Sofía V,José A,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
5,P001 Los Olivos - PVH,202501,Sebastián S,Vik E,88436.867727,4731755.0,53.504321,83390.117247,5157284.0,56.742390,79419.0,4992849.0,62.867056


**Productividad**

In [5]:
df_prod1["Periodo"] = df_prod1["Periodo"].astype(str)
df_prod2 = df_prod1[df_prod1["Periodo"]==periodo_analisis]
venta = df_prod2["VtaNeta"].sum()
jeq = df_prod2["JEq"].sum()
venta_meta = df_prod2["VtaNetaMeta"].sum()
jeq_meta = df_prod2["JEqMeta"].sum()
venta_aa = df_prod2["VtaNetaAA"].sum()
jeq_aa = df_prod2["JEqAA"].sum()


prod = venta/jeq
prod_meta = venta_meta/jeq_meta
prod_aa = venta_aa/jeq_aa

var_meta = prod/prod_meta
var_aa = prod/prod_aa

In [6]:
df_prod1["año"] = df_prod1["Periodo"].str[0:4]
df_prod1["mes"] = df_prod1["Periodo"].str[-2:]

dffx = df_prod1[(df_prod1["año"]=="2026")&(df_prod1["Periodo"]<=periodo_analisis)].groupby(["mes"]).agg({"VtaNeta":"sum", "JEq":"sum","VtaNetaMeta":"sum","JEqMeta":"sum"}).reset_index()
dffy = df_prod1[(df_prod1["año"]=="2025")].groupby(["mes"]).agg(VtaAA=("VtaNeta","sum"), JEqAA = ("JEq","sum")).reset_index()
dffz = dffy.merge(dffx, on="mes", how="left")
dffz["Prod"] = dffz["VtaNeta"] / dffz["JEq"]
dffz["ProdMeta"] = dffz["VtaNetaMeta"] / dffz["JEqMeta"]
dffz["ProdAA"] = dffz["VtaAA"] / dffz["JEqAA"]
dffz["Cumplimiento_Meta"] = dffz["Prod"]/dffz["ProdMeta"]-1
dffz

,mes,VtaAA,JEqAA,VtaNeta,JEq,VtaNetaMeta,JEqMeta,Prod,ProdMeta,ProdAA,Cumplimiento_Meta
0,01,518116745.0,6171.840867,533671400.0,6049.577070,547667319.0,6153.103817,88216.315593,89006.676194,83948.493840,-0.008880
1,02,490866961.0,6276.841719,530816787.0,6167.054740,521795910.0,6153.103817,86072.981255,84802.065030,78202.857901,0.014987
2,03,602709832.0,6282.130430,644100872.0,6396.902816,639876463.0,6153.103817,100689.488420,103992.469827,95940.356334,-0.031762
3,04,506226029.0,6065.014861,555617870.0,5971.190313,536322776.0,6153.103817,93049.767454,87162.965550,83466.576850,0.067538
4,05,520242043.0,6132.419577,571848102.0,6049.699523,552687262.0,6153.103817,94525.042085,89822.515346,84834.711079,0.052354
5,06,478564210.0,6018.327729,520496580.0,6001.593403,514703642.0,6153.103817,86726.398319,83649.432438,79517.804868,0.036784
6,07,513932345.0,5996.786297,NaN,NaN,NaN,NaN,NaN,NaN,85701.293917,NaN
7,08,500559595.0,5685.091351,NaN,NaN,NaN,NaN,NaN,NaN,88047.766362,NaN
8,09,460488523.0,5640.100132,NaN,NaN,NaN,NaN,NaN,NaN,81645.451717,NaN
9,10,475397032.0,5635.741163,NaN,NaN,NaN,NaN,NaN,NaN,84353.950666,NaN


In [7]:
df_gos = df_prod2.groupby("GOS").agg({"VtaNeta":"sum","JEq":"sum", "VtaNetaMeta":"sum","JEqMeta":"sum", "VtaNetaAA":"sum", "JEqAA":"sum"}).reset_index()
df_gos["Prod"] = df_gos["VtaNeta"] / df_gos["JEq"]
df_gos["ProdMeta"] = df_gos["VtaNetaMeta"] / df_gos["JEqMeta"]
df_gos["ProdAA"] = df_gos["VtaNetaAA"] / df_gos["JEqAA"]
df_gos["Cump_Meta"] = df_gos["Prod"]/df_gos["ProdMeta"]-1
df_gos["Cump_Meta2"] = df_gos["Prod"]/df_gos["ProdMeta"]
df_gos["Cump_AA"] = df_gos["Prod"]/df_gos["ProdAA"]-1
df_gos = df_gos.sort_values(by="Cump_Meta", ascending=False)
df_gos

,GOS,VtaNeta,JEq,VtaNetaMeta,JEqMeta,VtaNetaAA,JEqAA,Prod,ProdMeta,ProdAA,Cump_Meta,Cump_Meta2,Cump_AA
0,Fátima L,8484263.0,132.790972,8242587.0,142.260192,8351761.0,165.373785,63891.865976,57940.221080,50502.327283,0.102720,1.102720,0.265127
4,Sofía V,197488607.0,2090.142833,196293076.0,2229.046568,173930756.0,2064.874924,94485.699183,88061.451389,84233.070977,0.072952,1.072952,0.121717
1,Hernán P,55242020.0,703.509382,52149703.0,708.059011,52614602.0,726.377799,78523.501488,73651.633795,72434.209995,0.066147,1.066147,0.084067
3,Sebastián S,237468487.0,2659.271319,233492131.0,2682.391384,223085444.0,2689.712278,89298.329683,87046.257438,82940.263107,0.025872,1.025872,0.076658
2,María R,21813203.0,415.878896,24526145.0,391.346660,20581647.0,371.988944,52450.853406,62671.149330,55328.652390,-0.163078,0.836922,-0.052013


In [8]:
df_gor = df_prod2.groupby("GOR").agg({"VtaNeta":"sum","JEq":"sum", "VtaNetaMeta":"sum","JEqMeta":"sum", "VtaNetaAA":"sum", "JEqAA":"sum"}).reset_index()
df_gor["Prod"] = df_gor["VtaNeta"] / df_gor["JEq"]
df_gor["ProdMeta"] = df_gor["VtaNetaMeta"] / df_gor["JEqMeta"]
df_gor["ProdAA"] = df_gor["VtaNetaAA"] / df_gor["JEqAA"]
df_gor["Cump_Meta"] = df_gor["Prod"]/df_gor["ProdMeta"]-1
df_gor["Cump_Meta2"] = df_gor["Prod"]/df_gor["ProdMeta"]
df_gor["Cump_AA"] = df_gor["Prod"]/df_gor["ProdAA"]-1
df_gor = df_gor.sort_values(by="Cump_Meta", ascending=False)
df_gor

,GOR,VtaNeta,JEq,VtaNetaMeta,JEqMeta,VtaNetaAA,JEqAA,Prod,ProdMeta,ProdAA,Cump_Meta,Cump_Meta2,Cump_AA
6,Melany A,61306082.0,688.560688,62147928.0,800.715670,57062378.0,766.935896,89035.117910,77615.476189,74403.060686,0.147131,1.147131,0.196659
1,Fátima L,8484263.0,132.790972,8242587.0,142.260192,8351761.0,165.373785,63891.865976,57940.221080,50502.327283,0.102720,1.102720,0.265127
0,Daniel R,55242020.0,703.509382,52149703.0,708.059011,52614602.0,726.377799,78523.501488,73651.633795,72434.209995,0.066147,1.066147,0.084067
4,José A,67691322.0,697.529049,68140216.0,729.990429,56337212.0,583.009340,97044.448736,93343.985504,96631.748598,0.039643,1.039643,0.004271
7,Rodolfo O,81000651.0,897.826854,76654091.0,875.749054,75977015.0,916.522785,90218.565667,87529.744564,82897.028057,0.030719,1.030719,0.088321
2,Joe H,68491203.0,704.053097,66004932.0,698.340470,60531166.0,714.929688,97281.303456,94516.836524,84667.299538,0.029248,1.029248,0.148983
3,Johanna V,61843471.0,668.660319,61950178.0,689.166433,56625266.0,648.950604,92488.621205,89891.461725,87256.665818,0.028892,1.028892,0.059961
8,Vik E,94624365.0,1092.784146,94887862.0,1117.475898,90483163.0,1124.238889,86590.170036,84912.669872,80483.929078,0.019756,1.019756,0.075869
5,María R,21813203.0,415.878896,24526145.0,391.346660,20581647.0,371.988944,52450.853406,62671.149330,55328.652390,-0.163078,0.836922,-0.052013


In [9]:
df_tienda = df_prod2.groupby("Tienda").agg({"VtaNeta":"sum","JEq":"sum", "VtaNetaMeta":"sum","JEqMeta":"sum", "VtaNetaAA":"sum", "JEqAA":"sum"}).reset_index()
df_tienda["Prod"] = df_tienda["VtaNeta"] / df_tienda["JEq"]
df_tienda["ProdMeta"] = df_tienda["VtaNetaMeta"] / df_tienda["JEqMeta"]
df_tienda["ProdAA"] = df_tienda["VtaNetaAA"] / df_tienda["JEqAA"]
df_tienda["Cump_Meta"] = df_tienda["Prod"]/df_tienda["ProdMeta"]-1
df_tienda["Cump_Meta2"] = df_tienda["Prod"]/df_tienda["ProdMeta"]
df_tienda["Cump_AA"] = df_tienda["Prod"]/df_tienda["ProdAA"]-1
df_tienda = df_tienda.sort_values(by="Cump_Meta", ascending=False)
df_tienda1 = df_tienda[
    np.isfinite(df_tienda["Prod"]) &
    np.isfinite(df_tienda["ProdMeta"]) &
    (df_tienda["Prod"] > 1) &
    (df_tienda["ProdMeta"] > 1)
].copy()

regex_limpieza = r"^[A-Z0-9 ]+\s(?=[A-Z][a-z])|\s*-\s*[A-Z]+$"

df_tienda1["Tienda"] = df_tienda1["Tienda"].str.replace(
    regex_limpieza, "", regex=True
)
df_tienda1_topx = df_tienda1.sort_values(by="Cump_Meta", ascending=False)
df_tienda1_top = df_tienda1_topx[~df_tienda1_topx["Tienda"].isin(["Primavera", "Asia"])]


df_tienda1_bot = df_tienda1.sort_values(by="Cump_Meta", ascending=True)


In [10]:
df_prod3 = df_prod1[(df_prod1["año"]=="2026")&(df_prod1["Periodo"]<=periodo_analisis)]

df_gor_historico = df_prod3.groupby(["GOR", "mes"]).agg({"VtaNeta":"sum","JEq":"sum", "VtaNetaMeta":"sum","JEqMeta":"sum", "VtaNetaAA":"sum", "JEqAA":"sum"}).reset_index()
df_gor_historico["Prod"] = df_gor_historico["VtaNeta"] / df_gor_historico["JEq"]
df_gor_historico["ProdMeta"] = df_gor_historico["VtaNetaMeta"] / df_gor_historico["JEqMeta"]
df_gor_historico["ProdAA"] = df_gor_historico["VtaNetaAA"] / df_gor_historico["JEqAA"]
df_gor_historico["Cump_Meta"] = df_gor_historico["Prod"]/df_gor_historico["ProdMeta"]-1
df_gor_historico["Cump_Meta2"] = df_gor_historico["Prod"]/df_gor_historico["ProdMeta"]
df_gor_historico["Cump_AA"] = df_gor_historico["Prod"]/df_gor_historico["ProdAA"]-1
df_gor_historico = df_gor_historico.sort_values(by="Cump_Meta", ascending=False)


In [11]:
df_gor_bot = df_prod1[(df_prod1["Periodo"]==periodo_analisis)]
df_gor_bot["Prod"] = df_gor_bot["VtaNeta"] / df_gor_bot["JEq"]
df_gor_bot["ProdMeta"] = df_gor_bot["VtaNetaMeta"] / df_gor_bot["JEqMeta"]
df_gor_bot["ProdAA"] = df_gor_bot["VtaNetaAA"] / df_gor_bot["JEqAA"]
df_gor_bot["Cump_Meta"] = df_gor_bot["Prod"]/df_gor_bot["ProdMeta"]-1
df_gor_bot["Cump_AA"] = df_gor_bot["Prod"]/df_gor_bot["ProdAA"]-1
df_gor_bot["Tienda"] = df_gor_bot["Tienda"].str.replace(
    regex_limpieza, "", regex=True
)

lista_gor = df_gor_bot["GOR"].unique().tolist()


In [12]:
graff.KPI_Productividad_HTML(
    prod,
    "PRODUCTIVIDAD",
    prod_meta,
    prod_aa,
    var_meta,
    var_aa,
    r"html/kpi_productividad.html"
)

graff.Historico_Comparativo(
    df=dffz,
    anio1="ProdAA",
    anio2="Prod",
    cumplimiento="Cumplimiento_Meta",
    titulo = "Productividad -  Histórico",
    archivo_html=r"html/Historico_Productividad.html",
    ymin=None,
    ymax=None,
    y2min=-0.4,
    y2max=0.10,
    col_mes="mes"
)

graff.Ranking_HTML(
    df=df_gos,
    col_nombre="GOS",
    col_cumplimiento="Cump_Meta2",
    col_valor1="Cump_Meta",
    col_valor2="Cump_AA",
    col_valor3="Prod",
    titulo="Ranking GOS (▲ Meta | ▲ AA | Prod)",
    archivo_html=r"html/Ranking_GOS.html"
)

graff.Ranking_HTML(
    df=df_gor,
    col_nombre="GOR",
    col_cumplimiento="Cump_Meta2",
    col_valor1="Cump_Meta",
    col_valor2="Cump_AA",
    col_valor3="Prod",
    titulo="Ranking GOR (▲ Meta | ▲ AA | Prod)",
    archivo_html=r"html/Ranking_GOR.html"
)

graff.Ranking_HTML(
    df=df_tienda1_top.head(7),
    col_nombre="Tienda",
    col_cumplimiento="Cump_Meta2",
    col_valor1="Cump_Meta",
    col_valor2="Cump_AA",
    col_valor3="Prod",
    titulo="Tiendas TOP (▲ Meta | ▲ AA | Prod)",
    archivo_html=r"html/Ranking_TiendasTOP.html"
)

graff.Ranking_HTML(
    df=df_tienda1_bot.head(7),
    col_nombre="Tienda",
    col_cumplimiento="Cump_Meta2",
    col_valor1="Cump_Meta",
    col_valor2="Cump_AA",
    col_valor3="Prod",
    titulo="Tiendas BOT (▲ Meta | ▲ AA | Prod)",
    archivo_html=r"html/Ranking_TiendasBOT.html"
)

graff.generar_reporte_consolidado(
    kpi_path=r"html/kpi_productividad.html",
    historico_path=r"html/Historico_Productividad.html",
    gos_path=r"html/Ranking_GOS.html",
    gor_path=r"html/Ranking_GOR.html",
    top_path=r"html/Ranking_TiendasTOP.html",
    bot_path=r"html/Ranking_TiendasBOT.html",
    output_path=r"html/dashboard_consolidado.html")
graff.HTML_a_PNG(
    r"html/dashboard_consolidado.html",
    r"png/Dashboard_Productividad4.png")

graff.Heatmap_GOR(
    df_gor_historico,
    gor="GOR",
    mes="mes",
    valor="Cump_Meta",
    archivo_html=r"html/Heatmap_GOR.html",
    titulo="Cumplimiento vs Meta")
graff.HTML_a_PNG( r"html/Heatmap_GOR.html", r"png/Dashboard_historico_GOR.png")

graff.tiendas_top_bot_gor(df_gor_bot, lista_gor, r"html/productividad_tiendas_bot_gor.html", ascending=True)
graff.HTML_a_PNG( r"html/productividad_tiendas_bot_gor.html", r"png/productividad_tiendas_bot_gor.png")

graff.tiendas_top_bot_gor(df_gor_bot, lista_gor, r"html/productividad_tiendas_top_gor.html", ascending=False)
graff.HTML_a_PNG(r"html/productividad_tiendas_top_gor.html", r"png/productividad_tiendas_top_gor.png")

Archivo generado: html/Heatmap_GOR.html


'png/productividad_tiendas_top_gor.png'

**Velocidad**

In [13]:
df_veloc = pd.read_csv(ruta_velocdidad)

df_veloc["GOS"] = ( df_veloc["regional"].str.split().str[0]  + " " + df_veloc["regional"].str.split().str[1].str[0])
df_veloc["GOR"] = (df_veloc["supervisor"].str.title().str.split().str[0] + " " + df_veloc["supervisor"].str.title().str.split().str[1].str[0])

meses_dict = {
    "enero": "01",
    "febrero": "02",
    "marzo": "03",
    "abril": "04",
    "mayo": "05",
    "junio": "06",
    "julio": "07",
    "agosto": "08",
    "septiembre": "09",
    "octubre": "10",
    "noviembre": "11",
    "diciembre": "12"
}

df_veloc["mes"] = df_veloc["Mes"].str.lower().map(meses_dict)
df_veloc["año"] = df_veloc["Año"].astype(str)
df_veloc["Periodo"] = df_veloc["año"].astype(str) + df_veloc["mes"]
df_veloc["Tienda"] = df_veloc["nombre"].str.split(" - ").str[0]

df_aa = ( df_veloc[["Tienda", "mes", "año", "venta_unidad", "tiempo_escaneo"]].rename(columns={"venta_unidad": "venta_unidad_aa", "tiempo_escaneo": "tiempo_escaneo_aa"}))
df_aa["año"] = df_aa["año"].astype(int)
df_aa["año"] = df_aa["año"] + 1
df_aa["año"] = df_aa["año"].astype(str)

df_veloc1 = df_veloc.merge( df_aa, on=["Tienda", "mes", "año"], how="left")

In [14]:
df_veloc2 = df_veloc1[df_veloc1["Periodo"]==periodo_analisis]
df_veloc2

venta_unid = df_veloc2["venta_unidad"].sum()
tiempo_scan = df_veloc2["tiempo_escaneo"].sum()
venta_unid_aa = df_veloc2["venta_unidad_aa"].sum()
tiempo_scan_aa = df_veloc2["tiempo_escaneo_aa"].sum()


velocidad = venta_unid/tiempo_scan*60
velocidad_aa = venta_unid_aa/tiempo_scan_aa*60
velocidad_meta = df_veloc2["Objetivo"].mean()

var_veloc_meta = velocidad/velocidad_meta
var_veloc_aa = velocidad/velocidad_aa


In [15]:
df_veloc1x = df_veloc1[(df_veloc1["año"]=="2026")&(df_veloc1["Periodo"]<=periodo_analisis)].groupby(["mes"]).agg({"venta_unidad":"sum", "tiempo_escaneo":"sum","Objetivo":"mean"}).reset_index()
df_veloc1y = df_veloc1[(df_veloc1["año"]=="2025")].groupby(["mes"]).agg(venta_unidad_AA=("venta_unidad","sum"), tiempo_escaneo_AA = ("tiempo_escaneo","sum")).reset_index()
df_veloc1z = df_veloc1y.merge(df_veloc1x, on="mes", how="left")
df_veloc1z["Velocidad"] = df_veloc1z["venta_unidad"] / df_veloc1z["tiempo_escaneo"]*60
df_veloc1z["VelocidadMeta"] = df_veloc1z["Objetivo"]
df_veloc1z["VelocidadAA"] = df_veloc1z["venta_unidad_AA"] / df_veloc1z["tiempo_escaneo_AA"]*60
df_veloc1z["Cumplimiento_Meta"] = df_veloc1z["Velocidad"]/df_veloc1z["VelocidadMeta"]-1
df_veloc1z

,mes,venta_unidad_AA,tiempo_escaneo_AA,venta_unidad,tiempo_escaneo,Objetivo,Velocidad,VelocidadMeta,VelocidadAA,Cumplimiento_Meta
0,01,5.327118e+07,263212356.0,51071758.59,260697425.0,13.814159,11.754261,13.814159,12.143315,-0.149115
1,02,4.983687e+07,243574678.0,49019695.06,237332596.0,13.814159,12.392658,13.814159,12.276368,-0.102902
2,03,5.962989e+07,284186136.0,58228078.64,269503570.0,13.814159,12.963408,13.814159,12.589612,-0.061585
3,04,5.156991e+07,248547228.0,48629474.45,222641363.0,13.814159,13.105240,13.814159,12.449120,-0.051318
4,05,5.301125e+07,254613326.0,52304690.32,238226684.0,13.814159,13.173509,13.814159,12.492178,-0.046376
5,06,5.018963e+07,246690384.0,48278604.67,219292814.0,13.814159,13.209353,13.814159,12.207115,-0.043782
6,07,5.154568e+07,243262231.0,NaN,NaN,NaN,NaN,NaN,12.713608,NaN
7,08,5.125575e+07,242908280.0,NaN,NaN,NaN,NaN,NaN,12.660520,NaN
8,09,4.811315e+07,226561861.0,NaN,NaN,NaN,NaN,NaN,12.741726,NaN
9,10,4.989798e+07,230922315.0,NaN,NaN,NaN,NaN,NaN,12.964874,NaN


In [16]:
df_vel_gos = df_veloc2.groupby("GOS").agg({"venta_unidad":"sum","tiempo_escaneo":"sum", "Objetivo":"mean", "venta_unidad_aa":"sum", "tiempo_escaneo_aa":"sum"}).reset_index()
df_vel_gos["Velocidad"] = df_vel_gos["venta_unidad"] / df_vel_gos["tiempo_escaneo"]*60
df_vel_gos["VelocidadMeta"] = df_vel_gos["Objetivo"]
df_vel_gos["VelocidadAA"] = df_vel_gos["venta_unidad_aa"] / df_vel_gos["tiempo_escaneo_aa"]*60
df_vel_gos["Cump_Meta"] = df_vel_gos["Velocidad"]/df_vel_gos["VelocidadMeta"]-1
df_vel_gos["Cump_Meta2"] = df_vel_gos["Velocidad"]/df_vel_gos["VelocidadMeta"]
df_vel_gos["Cump_AA"] = df_vel_gos["Velocidad"]/df_vel_gos["VelocidadAA"]-1
df_vel_gos = df_vel_gos.sort_values(by="Cump_Meta", ascending=False)
df_vel_gos

,GOS,venta_unidad,tiempo_escaneo,Objetivo,venta_unidad_aa,tiempo_escaneo_aa,Velocidad,VelocidadMeta,VelocidadAA,Cump_Meta,Cump_Meta2,Cump_AA
2,María R,1827002.31,11681708.0,9.142857,1992441.47,13873864.0,9.383914,9.142857,8.616669,0.026366,1.026366,0.089042
0,Fátima L,1213989.85,5436421.0,13.857143,1155938.83,6128805.0,13.398409,13.857143,11.316452,-0.033105,0.966895,0.183976
4,Sofía V,17342292.61,72340044.0,14.914286,18337545.05,79741551.0,14.383977,14.914286,13.797734,-0.035557,0.964443,0.042488
1,Hernán P,6118312.22,29260262.0,13.125000,5874533.18,32236354.0,12.545982,13.125000,10.933991,-0.044116,0.955884,0.147429
3,Sebastián S,21777007.68,100574379.0,13.926829,22829173.37,114709810.0,12.991584,13.926829,11.941005,-0.067154,0.932846,0.087981


In [17]:
df_vel_gor = df_veloc2.groupby("GOR").agg({"venta_unidad":"sum","tiempo_escaneo":"sum", "Objetivo":"mean", "venta_unidad_aa":"sum", "tiempo_escaneo_aa":"sum"}).reset_index()
df_vel_gor["Velocidad"] = df_vel_gor["venta_unidad"] / df_vel_gor["tiempo_escaneo"]*60
df_vel_gor["VelocidadMeta"] = df_vel_gor["Objetivo"]
df_vel_gor["VelocidadAA"] = df_vel_gor["venta_unidad_aa"] / df_vel_gor["tiempo_escaneo_aa"]*60
df_vel_gor["Cump_Meta"] = df_vel_gor["Velocidad"]/df_vel_gor["VelocidadMeta"]-1
df_vel_gor["Cump_Meta2"] = df_vel_gor["Velocidad"]/df_vel_gor["VelocidadMeta"]
df_vel_gor["Cump_AA"] = df_vel_gor["Velocidad"]/df_vel_gor["VelocidadAA"]-1
df_vel_gor = df_vel_gor.sort_values(by="Cump_Meta", ascending=False)
df_vel_gor

,GOR,venta_unidad,tiempo_escaneo,Objetivo,venta_unidad_aa,tiempo_escaneo_aa,Velocidad,VelocidadMeta,VelocidadAA,Cump_Meta,Cump_Meta2,Cump_AA
4,María R,1827002.31,11681708.0,9.142857,1992441.47,13873864.0,9.383914,9.142857,8.616669,0.026366,1.026366,0.089042
6,Pepe A,5766455.52,23715263.0,14.909091,6050822.45,25039799.0,14.589226,14.909091,14.498892,-0.021454,0.978546,0.006230
1,Fátima L,1213989.85,5436421.0,13.857143,1155938.83,6128805.0,13.398409,13.857143,11.316452,-0.033105,0.966895,0.183976
2,Joe H,6015878.55,25503314.0,14.727273,5934643.68,27415506.0,14.153169,14.727273,12.988220,-0.038982,0.961018,0.089693
7,Rodolfo O,7082191.63,31561327.0,14.066667,7634050.95,37352032.0,13.463677,14.066667,12.262869,-0.042867,0.957133,0.097922
5,Melany A,5559958.54,23121467.0,15.076923,6352078.92,27286246.0,14.428043,15.076923,13.967650,-0.043038,0.956962,0.032961
0,Daniel R,6118312.22,29260262.0,13.125000,5874533.18,32236354.0,12.545982,13.125000,10.933991,-0.044116,0.955884,0.147429
8,Vik E,9293292.77,41301564.0,14.166667,9956996.43,48283874.0,13.500640,14.166667,12.373071,-0.047014,0.952986,0.091131
3,Johanna V,5401523.28,27711488.0,13.125000,5238125.99,29073904.0,11.695200,13.125000,10.809954,-0.108937,0.891063,0.081892


In [18]:
df_veloc_tienda = df_veloc2.groupby("Tienda").agg({"venta_unidad":"sum","tiempo_escaneo":"sum", "Objetivo":"mean", "venta_unidad_aa":"sum", "tiempo_escaneo_aa":"sum"}).reset_index()
df_veloc_tienda["Velocidad"] = df_veloc_tienda["venta_unidad"] / df_veloc_tienda["tiempo_escaneo"]*60
df_veloc_tienda["VelocidadMeta"] = df_veloc_tienda["Objetivo"]
df_veloc_tienda["VelocidadAA"] = df_veloc_tienda["venta_unidad_aa"] / df_veloc_tienda["tiempo_escaneo_aa"]*60
df_veloc_tienda["Cump_Meta"] = df_veloc_tienda["Velocidad"]/df_veloc_tienda["VelocidadMeta"]-1
df_veloc_tienda["Cump_Meta2"] = df_veloc_tienda["Velocidad"]/df_veloc_tienda["VelocidadMeta"]
df_veloc_tienda["Cump_AA"] = df_veloc_tienda["Velocidad"]/df_veloc_tienda["VelocidadAA"]-1
df_veloc_tienda = df_veloc_tienda.sort_values(by="Cump_Meta", ascending=False)

df_veloc_tienda_top = df_veloc_tienda.sort_values(by="Cump_Meta", ascending=False)
df_veloc_tienda_bot = df_veloc_tienda.sort_values(by="Cump_Meta", ascending=True)

In [19]:
df_veloc3 = df_veloc1[(df_veloc1["año"]=="2026")&(df_veloc1["Periodo"]<=periodo_analisis)]

df_gor_veloc_hist = df_veloc3.groupby(["GOR", "mes"]).agg({"venta_unidad":"sum","tiempo_escaneo":"sum", "Objetivo":"mean", "venta_unidad_aa":"sum", "tiempo_escaneo_aa":"sum"}).reset_index()
df_gor_veloc_hist["Velocidad"] = df_gor_veloc_hist["venta_unidad"] / df_gor_veloc_hist["tiempo_escaneo"]*60
df_gor_veloc_hist["VelocidadMeta"] = df_gor_veloc_hist["Objetivo"]
df_gor_veloc_hist["VelocidadAA"] = df_gor_veloc_hist["venta_unidad_aa"] / df_gor_veloc_hist["tiempo_escaneo_aa"]*60
df_gor_veloc_hist["Cump_Meta"] = df_gor_veloc_hist["Velocidad"]/df_gor_veloc_hist["VelocidadMeta"]-1
df_gor_veloc_hist["Cump_Meta2"] = df_gor_veloc_hist["Velocidad"]/df_gor_veloc_hist["VelocidadMeta"]
df_gor_veloc_hist["Cump_AA"] = df_gor_veloc_hist["Velocidad"]/df_gor_veloc_hist["VelocidadAA"]-1
df_gor_veloc_hist = df_gor_veloc_hist.sort_values(by="Cump_Meta", ascending=False)
df_gor_veloc_hist.head()

,GOR,mes,venta_unidad,tiempo_escaneo,Objetivo,venta_unidad_aa,tiempo_escaneo_aa,Velocidad,VelocidadMeta,VelocidadAA,Cump_Meta,Cump_Meta2,Cump_AA
29,María R,06,1827002.31,11681708.0,9.142857,1992441.470,13873864.0,9.383914,9.142857,8.616669,0.026366,1.026366,0.089042
28,María R,05,1949503.51,12727959.0,9.142857,2048567.460,14139822.0,9.190021,9.142857,8.692758,0.005159,1.005159,0.057204
25,María R,02,1744071.01,11593894.0,9.142857,2102530.538,14178704.0,9.025808,9.142857,8.897275,-0.012802,0.987198,0.014446
26,María R,03,2041998.70,13628629.0,9.142857,2237708.755,15464387.0,8.989893,9.142857,8.682046,-0.016730,0.983270,0.035458
27,María R,04,2049698.74,13681695.0,9.142857,2148072.177,14904034.0,8.988793,9.142857,8.647614,-0.016851,0.983149,0.039454


In [20]:
df_veloc_gor_bot = df_veloc1[(df_veloc1["Periodo"]==periodo_analisis)]
df_veloc_gor_bot["Velocidad"] = df_veloc_gor_bot["venta_unidad"] / df_veloc_gor_bot["tiempo_escaneo"]*60
df_veloc_gor_bot["VelocidadMeta"] = df_veloc_gor_bot["Objetivo"]
df_veloc_gor_bot["VelocidadAA"] = df_veloc_gor_bot["venta_unidad_aa"] / df_veloc_gor_bot["tiempo_escaneo_aa"]*60
df_veloc_gor_bot["Cump_Meta"] = df_veloc_gor_bot["Velocidad"]/df_veloc_gor_bot["VelocidadMeta"]-1
df_veloc_gor_bot["Cump_AA"] = df_veloc_gor_bot["Velocidad"]/df_veloc_gor_bot["VelocidadAA"]-1

lista_gor = df_veloc_gor_bot["GOR"].unique().tolist()


In [21]:
graff.KPI_Productividad_HTML(
    velocidad,
    "VELOCIDAD",
    velocidad_meta,
    velocidad_aa,
    var_veloc_meta,
    var_veloc_aa,
    r"html/kpi_velocidad.html"
)

graff.Historico_Comparativo(
    df=df_veloc1z,
    anio1="VelocidadAA",
    anio2="Velocidad",
    cumplimiento="Cumplimiento_Meta",
    titulo = "Velocidad -  Histórico",
    archivo_html=r"html/Historico_Velocidad.html",
    ymin=None,
    ymax=None,
    y2min=-0.8,
    y2max=0.10,
    col_mes="mes"
)

graff.Ranking_HTML(
    df=df_vel_gos,
    col_nombre="GOS",
    col_cumplimiento="Cump_Meta2",
    col_valor1="Cump_Meta",
    col_valor2="Cump_AA",
    col_valor3="Velocidad",
    titulo="Ranking GOS (▲ Meta | ▲ AA | Veloc)",
    archivo_html=r"html/Ranking_Velocidad_GOS.html"
)

graff.Ranking_HTML(
    df=df_vel_gor,
    col_nombre="GOR",
    col_cumplimiento="Cump_Meta2",
    col_valor1="Cump_Meta",
    col_valor2="Cump_AA",
    col_valor3="Velocidad",
    titulo="Ranking GOR (▲ Meta | ▲ AA | Veloc)",
    archivo_html=r"html/Ranking_Velocidad_GOR.html"
)

graff.Ranking_HTML(
    df=df_veloc_tienda_top.head(7),
    col_nombre="Tienda",
    col_cumplimiento="Cump_Meta2",
    col_valor1="Cump_Meta",
    col_valor2="Cump_AA",
    col_valor3="Velocidad",
    titulo="Tiendas TOP (▲ Meta | ▲ AA | Veloc)",
    archivo_html=r"html/Ranking_Velocidad_TiendasTOP.html"
)

graff.Ranking_HTML(
    df=df_veloc_tienda_bot.head(7),
    col_nombre="Tienda",
    col_cumplimiento="Cump_Meta2",
    col_valor1="Cump_Meta",
    col_valor2="Cump_AA",
    col_valor3="Velocidad",
    titulo="Tiendas BOT (▲ Meta | ▲ AA | Veloc)",
    archivo_html=r"html/Ranking_Velocidad_TiendasBOT.html"
)

graff.generar_reporte_consolidado(
    kpi_path=r"html/kpi_velocidad.html",
    historico_path=r"html/Historico_Velocidad.html",
    gos_path=r"html/Ranking_Velocidad_GOS.html",
    gor_path=r"html/Ranking_Velocidad_GOR.html",
    top_path=r"html/Ranking_Velocidad_TiendasTOP.html",
    bot_path=r"html/Ranking_Velocidad_TiendasBOT.html",
    output_path=r"html/dashboard_velocidad_consolidado.html")
graff.HTML_a_PNG(
    r"html/dashboard_velocidad_consolidado.html",
    r"png/Dashboard_Velocidad.png")

graff.Heatmap_GOR(
    df_gor_veloc_hist,
    gor="GOR",
    mes="mes",
    valor="Cump_Meta",
    archivo_html=r"html/Heatmap_Velocidad_GOR.html",
    titulo="Cumplimiento vs Meta")
graff.HTML_a_PNG( r"html/Heatmap_Velocidad_GOR.html", r"png/Heatmap_Velocidad_GOR.png")

graff.tiendas_top_bot_gor(df_veloc_gor_bot, lista_gor, r"html/velocidad_tiendas_bot_gor.html", ascending=True)
graff.HTML_a_PNG( r"html/velocidad_tiendas_bot_gor.html", r"png/velocidad_tiendas_bot_gor.png")

graff.tiendas_top_bot_gor(df_veloc_gor_bot, lista_gor, r"html/velocidad_tiendas_top_gor.html", ascending=False)
graff.HTML_a_PNG(r"html/velocidad_tiendas_top_gor.html", r"png/velocidad_tiendas_top_gor.png")

Archivo generado: html/Heatmap_Velocidad_GOR.html


'png/velocidad_tiendas_top_gor.png'

In [25]:
print("finish")

finish
